**Dataset Setup**

In [3]:
!unzip NEU_Dataset.zip -d /content/NEU_Dataset
!ls /content/NEU_Dataset

In [4]:
import yaml
with open('/content/NEU_Dataset/data.yaml','r') as f:
  data = yaml.safe_load(f)

data['path'] = '/content/NEU_Dataset'
print(data)

In [5]:
with open('/content/NEU_Dataset/data.yaml','w') as f:
  yaml.dump(data,f)

**Installing Dependencies**

In [6]:
!pip install ultralytics -q


**Model Training**

In [7]:
from ultralytics import YOLO
import yaml
with open('/content/NEU_Dataset/data.yaml') as f:
  cfg = yaml.safe_load(f)

print("Classes", cfg['nc'])
print("Names",cfg['names'])
print("Path", cfg['path'])

import os
train_images = len(os.listdir('/content/NEU_Dataset/train/images'))
val_images = len(os.listdir('/content/NEU_Dataset/valid/images'))
print(f"Train Images: {train_images}, Val Images: {val_images}")

In [8]:
model = YOLO('yolov8m.pt')
results = model.train(
    data = '/content/NEU_Dataset/data.yaml',
    epochs = 100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    save=True,
    project = '/content/runs',
    name='defect_v1',
    verbose=True
)

print("Training COmplete")
print(f"Best Model saved at {results.save_dir}/weights/best.pt")

**Results and Evaluations**

In [9]:
import pandas as pd
df = pd.read_csv('/content/runs/defect_v1/results.csv')

print(f"Total epochs run: {len(df)}")
print(f"Best mAP@50: {df['metrics/mAP50(B)'].max():.3f}")
print(f"Best precision: {df['metrics/precision(B)'].max():.3f}")
print(f"Best Recall: {df['metrics/recall(B)'].max():.3f}")


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    '/content/runs/defect_v1/weights/best.pt',
    '/content/drive/MyDrive/best_neu_defect.pt'
)
print("Model saved to Google Drive")

In [11]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df['train/box_loss'], label='Train', color='#00e5ff')
axes[0].plot(df['val/box_loss'],   label='Val',   color='#ff5e3a')
axes[0].set_title('Box Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df['train/cls_loss'], label='Train', color='#00e5ff')
axes[1].plot(df['val/cls_loss'],   label='Val',   color='#ff5e3a')
axes[1].set_title('Class Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(df['metrics/mAP50(B)'], color='#34d399')
axes[2].set_title('mAP@50')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [12]:
import os
path = '/content/runs/defect_v1/weights/best.pt'
print("Model exists:", os.path.exists(path))

**Defect Detection System**

In [13]:
import cv2 as cv
import glob
from ultralytics import YOLO
from google.colab.patches import cv2_imshow


model = YOLO('/content/runs/defect_v1/weights/best.pt')

val_images = glob.glob('/content/NEU_Dataset/valid/images/*.jpg')
print(f"Total val images : {len(val_images)}")

for img_path in val_images[:10]:
  frame = cv.imread(img_path)
  results = model(frame, verbose=False, conf=0.3)
  r = results[0]

  for box in r.boxes:
    conf = float(box.conf[0])
    cls = int(box.cls[0])
    label = model.names[cls]
    x1,y1,x2,y2 = map(int,box.xyxy[0])

    text_y = y1 - 10 if y1 > 20 else y1 + 25

    cv.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),3)
    cv.putText(frame, f'{label}{conf:.2f}',
               (x1, text_y), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

  print(f"Images {img_path.split('/')[-1]} | Detections {len(r.boxes)}")
  cv2_imshow(frame)
  print("______")

**Creating a test video file**

In [14]:
import cv2 as cv
import glob
import numpy as np

# Get all validation images
val_images = glob.glob('/content/NEU_Dataset/valid/images/*.jpg')
print(f"Total images: {len(val_images)}")

# Video settings
out = cv.VideoWriter(
    '/content/neu_defect_test.mp4',
    cv.VideoWriter_fourcc(*'mp4v'),
    2,           # 2 FPS — slow enough to see each image clearly
    (640, 640)
)

for img_path in val_images:
    frame = cv.imread(img_path)
    frame = cv.resize(frame, (640, 640))
    out.write(frame)

out.release()
print("Video created — /content/neu_defect_test.mp4")

**ROI zone Inspection**

In [15]:
import cv2 as cv
import numpy as np
from ultralytics import YOLO
from google.colab.patches import cv2_imshow
import sqlite3
from datetime import datetime

# ── MODEL ──
model = YOLO('/content/runs/defect_v1/weights/best.pt')

# ── SETTINGS ──
CONFIDENCE_THRESHOLD = 0.5
CRITICAL_CLASSES     = ['inclusion', 'pitted_surface']
DISPLAY_WIDTH        = 960   # resize for processing
DISPLAY_HEIGHT       = 540

# ── ROI — updated for 960x540 display size ──
ROI_POINTS = np.array([[50,  50],
                        [590, 50],
                        [590, 590],
                        [50,  590]])

# ── DATABASE ──
conn   = sqlite3.connect('/content/defect_log.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS inspections (
        id         INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp  TEXT,
        defect     TEXT,
        confidence REAL,
        decision   TEXT
    )
''')
conn.commit()

# ── COUNTERS ──
part_count   = 0
reject_count = 0

# ── MAIN LOOP ──
cap = cv.VideoCapture('/content/neu_defect_test.mp4')

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # RESIZE — this is the key fix for 4K video
    frame = cv.resize(frame, (DISPLAY_WIDTH, DISPLAY_HEIGHT))

    # Draw ROI zone
    cv.polylines(frame, [ROI_POINTS], isClosed=True,
                 color=(0, 255, 255), thickness=2)

    results = model(frame, verbose=False, conf=CONFIDENCE_THRESHOLD)
    r       = results[0]

    part_count += 1

    for box in r.boxes:
        conf  = float(box.conf[0])
        cls   = int(box.cls[0])
        label = model.names[cls]
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cx, cy = (x1+x2)//2, (y1+y2)//2

        inside = cv.pointPolygonTest(ROI_POINTS, (cx, cy), False)

        if inside >= 0:
            if label in CRITICAL_CLASSES and conf > 0.7:
                color    = (0, 0, 255)
                decision = 'REJECT'
                reject_count += 1
            else:
                color    = (0, 165, 255)
                decision = 'FLAG'

            cv.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            text_y = y1 - 10 if y1 > 20 else y1 + 25
            cv.putText(frame, f'{label} {conf:.2f} -> {decision}',
                       (x1, text_y), cv.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            cursor.execute('''
                INSERT INTO inspections (timestamp, defect, confidence, decision)
                VALUES (?, ?, ?, ?)
            ''', (datetime.now().isoformat(), label, conf, decision))
            conn.commit()

    # Stats overlay
    defect_rate = (reject_count / part_count * 100) if part_count > 0 else 0
    cv.putText(frame,
               f'Parts: {part_count} | Rejects: {reject_count} | Rate: {defect_rate:.1f}%',
               (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # Show every 15th frame
    if frame_count % 15 == 0:
        cv2_imshow(frame)

    if frame_count >= 150:
        break

cap.release()
conn.close()

print(f"\nDone")
print(f"Frames processed : {frame_count}")
print(f"Parts counted    : {part_count}")
print(f"Rejects          : {reject_count}")
print(f"Defect rate      : {defect_rate:.1f}%")

In [16]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/content/defect_log.db')

df = pd.read_sql_query("SELECT * FROM inspections", conn)
conn.close()

print(f"Total logged detections: {len(df)}")
print(f"\nDecision breakdown:")
print(df['decision'].value_counts())
print(f"\nDefect type breakdown:")
print(df['defect'].value_counts())
print(f"\nAverage confidence: {df['confidence'].mean():.2f}")
print(f"\nLast 5 entries:")
print(df.tail())